# COVID-19 Exercise - Lane B (backup: the agent's output, captured)

**SISMID 2026 - Day 2, 3:30.** Continues Mauricio's exercise 3: use digital traces to
detect when a COVID-19 outbreak is *taking off* in Washington State.

> Each cell is a captured example of what a coding agent (Codex, Claude Code, or Antigravity
> CLI) produces from the **Lane A** prompts. Run it top to bottom as a backup, or read it
> as the target. Data: `covid_traces_WA.csv` (pre-smoothed daily; `new_cases` = ground
> truth, plus five digital traces).


## Step 0: load and look (6-panel)


In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, os

CANDIDATES = ['../data/covid_traces_WA.csv', 'data/covid_traces_WA.csv', './covid_traces_WA.csv']
path = next((p for p in CANDIDATES if os.path.exists(p)), CANDIDATES[0])
df = pd.read_csv(path); df['date'] = pd.to_datetime(df['date'])
TRACES = ['new_cases','upToDate','cdc_ili','Twitter_RelatedTweets','google_fever','Kinsa_AnomalousFeverAbsolute']
df.head()


In [ ]:
# --- Produced by the agent from the Plan A / Step 0 prompt ---
fig, axes = plt.subplots(3, 2, figsize=(11, 7))
for ax, col in zip(axes.ravel(), TRACES):
    ax.plot(df['date'], df[col]); ax.set_title(col, fontsize=9)
plt.tight_layout(); plt.show()


## Step 1: the growth factor alpha

Slide an 11-day window; regress the last 10 days on the previous 10 (no intercept). The
slope alpha is the multiplicative growth factor. alpha > 1 means growing.


In [ ]:
def growth_alpha(x):
    """Sliding 11-day growth factor: regress the last 10 days on the previous 10
    (through the origin). alpha > 1 means exponential growth."""
    x = np.asarray(x, float); n = len(x); alpha = np.zeros(n)
    for i in range(10, n):
        before = x[i-10:i]      # previous 10 days
        after  = x[i-9:i+1]     # next 10 days
        denom = np.sum(before*before)
        alpha[i] = np.sum(before*after)/denom if denom > 0 else 0.0
    return alpha


In [ ]:
# --- Produced by the agent from the Plan A / Step 1 prompt ---
alpha = growth_alpha(df['new_cases'])
fig, ax = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
ax[0].plot(df['date'], df['new_cases']); ax[0].set_ylabel('new_cases')
ax[1].plot(df['date'], alpha, 'b'); ax[1].axhline(1, color='r', lw=1); ax[1].set_ylabel('alpha')
ax[2].plot(df['date'], (alpha > 1).astype(int), 'g'); ax[2].set_ylabel('alpha > 1')
plt.tight_layout(); plt.show()


## Step 2: detect outbreak starts

Sustained growth (alpha>1 for 10 days) starts an outbreak; 10 flat days end it.


In [ ]:
def detect_outbreaks(alpha, run=10):
    """Mark an outbreak start after `run` consecutive days of alpha>1 (first day only);
    reset after `run` consecutive days of alpha<1."""
    n = len(alpha); starts = np.zeros(n, int); active = False
    for i in range(run, n):
        grow = alpha[i-run+1:i+1] > 1
        flat = alpha[i-run+1:i+1] < 1
        if grow.all():
            if not active: active = True; starts[i] = 1
        elif flat.all():
            active = False
    return starts


In [ ]:
# --- Produced by the agent from the Plan A / Step 2 prompt ---
starts = detect_outbreaks(alpha)
loc = np.where(starts == 1)[0]
print('detected outbreak starts:')
for i in loc: print('  ', df['date'].iloc[i].date())
plt.figure(figsize=(10,4))
plt.plot(df['date'], df['new_cases'], 'k')
plt.plot(df['date'].iloc[loc], df['new_cases'].iloc[loc], 'rx', ms=10, label='outbreak start')
plt.legend(); plt.title('Detected COVID-19 outbreak starts, WA'); plt.tight_layout(); plt.show()


## Step 3: multi-trace early warning

Run the same detector on every trace, and measure how many days each trace's outbreak
leads (negative) or lags (positive) the case-count outbreak.


In [ ]:
# --- Produced by the agent from the Plan A / Step 3 prompt ---
case_start = np.where(detect_outbreaks(growth_alpha(df['new_cases'])) == 1)[0]
first_case = case_start[0] if len(case_start) else None
print(f'{"trace":30s} first outbreak start   lead/lag vs cases (days)')
for col in TRACES:
    s = np.where(detect_outbreaks(growth_alpha(df[col])) == 1)[0]
    if len(s):
        lead = (s[0] - first_case) if first_case is not None else float('nan')
        print(f'{col:30s} {str(df["date"].iloc[s[0]].date()):20s} {lead:+d}')
    else:
        print(f'{col:30s} {"(none detected)":20s}')


## Reflection

- alpha turns ``is it taking off?'' into a number you can watch daily; the state machine
  turns that into an outbreak alarm.
- A trace with a consistently \textbf{negative} lead is an early-warning candidate.
- This onset detector complements ARGO: ARGO says how high, this says it is accelerating.

**Stretch:** tune the window length and the consecutive-day threshold; watch how the
detected starts move. Too sensitive fires on noise; too strict fires late.
